# 12. 3D Isosurfaces, Phonons, Wulff Shapes & Comic Strips

Explore `xkcd-matchem`'s flagship computational-materials and science-communication workflows — all running zero-dependency on NumPy, SciPy, and Matplotlib:
- **3D Orbital & Density Isosurfaces** (`xm.plot_isosurface` / `xm.extract_isosurface`) using Kuhn 6-tetrahedra Marching Tetrahedra.
- **Phonon Band Dispersions** (`xm.plot_phonon_bands`) from Phonopy `band.yaml` files with automatic imaginary-mode highlights.
- **Equilibrium 3D Wulff Crystal Shapes** (`xm.plot_wulff_shape` / `xm.compute_wulff_shape`).
- **2D Stereobonds & Reaction Schemes** (`xm.plot_reaction_scheme`), **3D Vectors**, and **Defect Callouts** (`xm.draw_defect_site`).
- **Narrative Comic Strips** (`xm.plot_comic_strip`, `draw_speech_bubble`, `draw_stick_scientist`).

## 1. 3D Orbital Isosurfaces & Phonon Dispersions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import xkcd_matchem as xm


q_pts = np.linspace(0.0, 1.0, 41)
soft_branch = 2.5 * np.sin(np.pi * q_pts) ** 2 - 1.4
opt_branch = 6.0 + 1.8 * np.cos(np.pi * q_pts)
ph_soft = xm.PhononBandStructure(
    q_distances=q_pts,
    frequencies=np.column_stack([soft_branch, opt_branch]),
    q_labels=["G", "X", "M"],
    q_ticks=[0.0, 0.5, 1.0],
)

with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.5))
    xm.plot_isosurface(
        "water_homo.cube",
        shade="crosshatch",
        ax=axes[0],
        title="Water HOMO (Cross-Hatch)",
    )
    xm.plot_phonon_bands(
        "si_phonon_bands.yaml",
        ax=axes[1],
        title="Silicon Phonon Dispersion (THz)",
    )
    xm.plot_phonon_bands(
        ph_soft,
        ax=axes[2],
        title="Soft Phase with Imaginary Modes",
    )
    fig.tight_layout()

## 1b. xkcd Pen-and-Ink 3D Shading Techniques

Compare all 8 depth-occluded shading styles side by side: `"solid"`, `"hatch"`, `"crosshatch"`, `"contour"`, `"cross_contour"`, `"stipple"`, `"scumble"`, and `"outline"`.

In [ ]:
modes = (
    "solid",
    "hatch",
    "crosshatch",
    "contour",
    "cross_contour",
    "stipple",
    "scumble",
    "outline",
)
with xm.xkcd_style("classic"):
    fig, axes = plt.subplots(2, 4, figsize=(15.2, 7.6))
    for ax, mode in zip(axes.ravel(), modes, strict=True):
        xm.plot_isosurface("water_homo.cube", shade=mode, ax=ax, title=mode)
    fig.tight_layout()

## 2. Equilibrium 3D Wulff Crystal Shapes & Surface Energies

In [ ]:
with xm.xkcd_style("classic"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.2, 4.8))
    xm.plot_wulff_shape(
        "si_diamond.cif",
        {(1, 0, 0): 1.25, (1, 1, 0): 1.15, (1, 1, 1): 0.88},
        ax=ax1,
        title="Si Wulff Shape ({111} Dominated)",
    )
    xm.plot_wulff_shape(
        "nacl.cif",
        {(1, 0, 0): 0.75, (1, 1, 1): 1.35},
        ax=ax2,
        title="Rock-Salt NaCl Cube ({100})",
    )
    fig.tight_layout()

## 3. Reaction Schemes, 3D Vectors, Defects & Multi-Panel Comic Strips

In [ ]:
si_atoms = xm.load("si_diamond.cif")
spins = np.zeros((len(si_atoms), 3), dtype=np.float64)
spins[:, 2] = np.array([1.0 if i % 2 == 0 else -1.0 for i in range(8)])

with xm.xkcd_style("classic"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.4, 4.4))
    xm.plot_reaction_scheme(
        ["benzene.sdf"],
        ["aspirin.xyz"],
        conditions_above="Carboxylation",
        conditions_below=r"$\mathrm{AlCl_3},\ 85\,^\circ\mathrm{C}$",
        ax=ax1,
        title="2D Skeletal Reaction Scheme",
    )
    xm.plot_crystal(
        si_atoms,
        vectors=spins,
        vector_scale=1.1,
        vector_color="spin",
        boundary_atoms=False,
        ax=ax2,
        title="AFM Spins + Vacancy Callout",
    )
    xm.draw_defect_site(
        ax2,
        si_atoms,
        (0.0, 0.0, 0.0),
        kind="vacancy",
        label="V_Si",
    )
    fig.tight_layout()

In [ ]:
strip = xm.plot_comic_strip(
    [
        xm.ComicPanel(
            title="1. Hypothesis",
            caption="Morning DFT run",
        ),
        xm.ComicPanel(
            title="2. Imaginary Modes?!",
            caption="Check q-mesh & supercell",
        ),
        xm.ComicPanel(
            title="3. Stable Crystal!",
            caption="All branches >= 0 THz",
        ),
    ],
    title="Adventures in Harmonic Phonons",
)
strip.add_stick_scientist(
    0,
    xy=(0.24, 0.14),
    character="ponytail",
    pose="think",
    prop="flask",
    speech="Cool TiO2\nand check phonons?",
)
xm.plot_phonon_bands(ph_soft, ax=strip.panel(1), title="Soft Mode Found")
xm.plot_phonon_bands(
    "si_phonon_bands.yaml", ax=strip.panel(2), title="Dynamically Stable"
)
strip.add_stick_scientist(
    2,
    xy=(0.82, 0.10),
    character="cueball",
    facing="left",
    pose="cheer",
    speech="Ship it!",
)

:::{tip}
Use `preset="talk_16x9"`, `preset="talk_dark"`, or `preset="graphical_abstract"` for slide decks and TOC art, and set `XM_COMMERCIAL_SAFE=1` for commercial-safe SIL OFL 1.1 typography everywhere!
:::